# 29 Data Quality Checks

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Build reusable data quality checks in PySpark (completeness, uniqueness, validity, ranges, referential integrity, freshness, volume and schema), run them in <b>one pass</b> where possible, produce a results table, and decide per check whether to <b>fail</b>, <b>warn</b> or <b>quarantine</b>.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Bad data that flows silently into Gold tables is worse than a failed job: dashboards show wrong numbers and nobody knows. Data quality checks are the guard rails between layers. The same ideas appear later as <b>Delta constraints</b> (<code>CHECK</code>, <code>NOT NULL</code>) and as <b>expectations</b> in Lakeflow Declarative Pipelines, which are on the certification exam. Writing your own checks first makes those features easy to understand.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Dimensions of data quality</b><br>
| Dimension | Question | Example check |<br>|---|---|---|<br>| Completeness | Are required values present? | <code>customer_id IS NOT NULL</code> |<br>| Uniqueness | Is the key unique? | count of <code>order_id</code> = distinct count |<br>| Validity | Do values match the rules? | <code>email</code> matches a pattern, <code>status</code> in an allowed list |<br>| Accuracy / range | Are values plausible? | <code>amount</code> between 0 and 100,000 |<br>| Consistency / integrity | Do references exist? | every <code>customer_id</code> exists in <code>customers</code> |<br>| Timeliness / freshness | Is the data recent enough? | latest <code>order_ts</code> within the last 24 hours |<br>| Volume | Is the row count normal? | between 80% and 120% of yesterday |<br>| Schema | Are the columns and types as expected? | schema matches the contract |<br><br>
<b>Actions</b> when a check fails:
<ul><li><b>Fail</b>: stop the pipeline (keys, schema, empty input)</li><li><b>Warn</b>: log and alert, but continue (small drift)</li><li><b>Quarantine</b>: move bad rows aside and continue with the good rows</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A logistics company's Silver <code>shipments</code> table feeds billing. Checks run before every write: <code>shipment_id</code> must be unique (<b>fail</b>), <code>weight_kg</code> must be between 0 and 30,000 (<b>quarantine</b> violators), and the row count must be within 50% of the 7-day average (<b>warn</b>). One day a partner duplicated a file. The uniqueness check stopped the load before customers were billed twice.
</div>

## Syntax

```python
from pyspark.sql import functions as F

# One pass: many checks as aggregate expressions
df.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("id").isNull().cast("int")).alias("null_ids"),
    F.countDistinct("id").alias("distinct_ids"),
    F.sum((~F.col("amount").between(0, 100000)).cast("int")).alias("out_of_range"),
)

# Row-level rule: flag, then split
flagged = df.withColumn("is_valid", F.col("id").isNotNull() & F.col("amount").between(0, 100000))
good, bad = flagged.filter("is_valid"), flagged.filter("NOT is_valid")
```

## Setup: orders and customers with problems

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates an orders table with a duplicate key, nulls, an invalid email, a negative amount, an unknown status and an unknown customer, plus a customers reference table.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
10 orders and 4 customers.
</div>

In [0]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

now = datetime(2024, 11, 10, 9, 0, 0)       # a fixed "current time" keeps results reproducible

orders = spark.createDataFrame(
    [
        ("O1", "C1", "amal@mail.com", 120.0, "PAID", now - timedelta(hours=2)),
        ("O2", "C2", "john@mail.co.uk", 80.0, "PAID", now - timedelta(hours=3)),
        ("O3", "C3", "vivek@mail", 45.0, "SHIPPED", now - timedelta(hours=5)),
        ("O4", "C1", "amal@mail.com", -15.0, "REFUND", now - timedelta(hours=6)),
        ("O5", None, "ghost@mail.com", 60.0, "PAID", now - timedelta(hours=7)),
        ("O6", "C4", "sara@mail.ae", 250000.0, "PAID", now - timedelta(hours=8)),
        ("O6", "C4", "sara@mail.ae", 250000.0, "PAID", now - timedelta(hours=8)),
        ("O7", "C9", "unknown@mail.com", 30.0, "LOST", now - timedelta(hours=9)),
        ("O8", "C2", None, 99.0, "PAID", now - timedelta(hours=10)),
        ("O9", "C3", "vivek@mail.ae", 75.0, "CANCELLED", now - timedelta(hours=11)),
    ],
    "order_id STRING, customer_id STRING, email STRING, amount DOUBLE, status STRING, order_ts TIMESTAMP",
)
customers = spark.createDataFrame([("C1",), ("C2",), ("C3",), ("C4",)], "customer_id STRING")
orders.show(truncate=False)

## 1. A data profile in one pass

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Computes row count, null counts, distinct counts, and min/max for every column in a <b>single</b> aggregation.<br><br>
<b>Why it matters</b><br>Profiling is the first step with any new dataset. Doing it in one <code>agg</code> means one scan, instead of a separate job per column and per metric.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
One row per column with <code>nulls</code>, <code>distinct</code> and min/max values. <code>customer_id</code> and <code>email</code> each have 1 null, and <code>order_id</code> has 9 distinct values in 10 rows.
</div>

In [0]:
def profile(df):
    exprs = []
    for c in df.columns:
        exprs += [
            F.sum(F.col(c).isNull().cast("int")).alias(f"{c}__nulls"),
            F.countDistinct(c).alias(f"{c}__distinct"),
            F.min(c).cast("string").alias(f"{c}__min"),
            F.max(c).cast("string").alias(f"{c}__max"),
        ]
    stats = df.agg(F.count("*").alias("rows"), *exprs).first().asDict()
    rows = [(c, stats["rows"], stats[f"{c}__nulls"], stats[f"{c}__distinct"], stats[f"{c}__min"], stats[f"{c}__max"])
            for c in df.columns]
    return spark.createDataFrame(rows, "column STRING, rows LONG, nulls LONG, distinct LONG, min STRING, max STRING")

profile(orders).show(truncate=False)

## 2. A reusable check library

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines small functions that each return a <b>check result</b> (name, failed row count, total, pass/fail, severity). Each check is a single aggregation.<br><br>
<b>Why it matters</b><br>Reusable checks turn data quality into code you can test and apply to any table. A standard result format makes it easy to log results, alert on them and store them over time.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No output yet: the functions are only defined.
</div>

In [0]:
from dataclasses import dataclass

@dataclass
class CheckResult:
    check: str
    column: str
    failed_rows: int
    total_rows: int
    severity: str        # "fail", "warn" or "quarantine"

    @property
    def passed(self):
        return self.failed_rows == 0

def _count_where(df, condition):
    """Number of rows where condition is true (null counts as false)."""
    return df.agg(F.sum(F.when(condition, 1).otherwise(0))).first()[0] or 0

def check_not_null(df, col, severity="fail"):
    return CheckResult("not_null", col, _count_where(df, F.col(col).isNull()), df.count(), severity)

def check_unique(df, col, severity="fail"):
    total = df.count()
    dupes = total - df.select(col).distinct().count()
    return CheckResult("unique", col, dupes, total, severity)

def check_in_range(df, col, low, high, severity="quarantine"):
    return CheckResult(f"range[{low},{high}]", col, _count_where(df, ~F.col(col).between(low, high)), df.count(), severity)

def check_allowed_values(df, col, allowed, severity="warn"):
    return CheckResult("allowed_values", col, _count_where(df, ~F.col(col).isin(allowed)), df.count(), severity)

def check_regex(df, col, pattern, severity="quarantine"):
    return CheckResult("regex", col, _count_where(df, F.col(col).isNotNull() & ~F.col(col).rlike(pattern)), df.count(), severity)

def check_foreign_key(df, col, ref_df, ref_col, severity="quarantine"):
    orphans = df.filter(F.col(col).isNotNull()).join(ref_df, df[col] == ref_df[ref_col], "left_anti").count()
    return CheckResult("foreign_key", col, orphans, df.count(), severity)

## 3. Run the checks and build a results table

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs a list of checks against <code>orders</code> and turns the results into a DataFrame.<br><br>
<b>Why it matters</b><br>A results table is easy to display, save to a Delta table for trend monitoring, and use to decide what happens next.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
7 checks, 6 of them failing: <code>order_id</code> has 1 duplicate, <code>customer_id</code> has 1 null, 3 amounts are out of range (<code>-15</code> and two copies of <code>250000</code>), 1 email is invalid (<code>vivek@mail</code>), 1 status isn't allowed (<code>LOST</code>), and 1 order references an unknown customer (<code>C9</code>). The <code>order_ts</code> not-null check passes.
</div>

In [0]:
EMAIL = r"^[\w.+-]+@[\w-]+\.[\w.]+$"
STATUSES = ["PAID", "SHIPPED", "REFUND", "CANCELLED"]

results = [
    check_unique(orders, "order_id"),
    check_not_null(orders, "customer_id"),
    check_in_range(orders, "amount", 0, 100_000),
    check_regex(orders, "email", EMAIL),
    check_allowed_values(orders, "status", STATUSES),
    check_foreign_key(orders, "customer_id", customers, "customer_id"),
    check_not_null(orders, "order_ts"),
]

results_df = spark.createDataFrame(
    [(r.check, r.column, r.failed_rows, r.total_rows, r.passed, r.severity) for r in results],
    "check STRING, column STRING, failed_rows LONG, total_rows LONG, passed BOOLEAN, severity STRING",
).withColumn("checked_at", F.lit(now))
results_df.show(truncate=False)

## 4. Freshness, volume and schema checks

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Checks that the newest record is recent enough, that the row count is within an expected range, and that the schema matches a contract.<br><br>
<b>Why it matters</b><br>These <b>table-level</b> checks catch problems that row-level rules miss: a source that stopped sending data, a half-delivered file, or a renamed column.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Freshness passes (newest order is 2 hours old, limit 24). Volume warns, because 10 rows is below the expected minimum of 12. The schema check reports <code>amount</code> as <code>double</code> while the contract expects <code>decimal(10,2)</code>.
</div>

In [0]:
# Freshness: how old is the newest record?
latest = orders.agg(F.max("order_ts")).first()[0]
age_hours = (now - latest).total_seconds() / 3600
print(f"freshness: newest record is {age_hours:.1f} h old -> {'OK' if age_hours <= 24 else 'STALE'}")

# Volume: compare with an expected range (for example from yesterday's count)
expected_min, expected_max = 12, 1000
n = orders.count()
print(f"volume: {n} rows -> {'OK' if expected_min <= n <= expected_max else 'WARN: outside expected range'}")

# Schema: compare with a contract
contract = {"order_id": "string", "customer_id": "string", "email": "string",
            "amount": "decimal(10,2)", "status": "string", "order_ts": "timestamp"}
actual = dict(orders.dtypes)
schema_issues = [f"{c}: expected {t}, got {actual.get(c, 'MISSING')}" for c, t in contract.items() if actual.get(c) != t]
extra = [c for c in actual if c not in contract]
print("schema issues:", schema_issues or "none", "| unexpected columns:", extra or "none")

## 5. Act on the results: fail, warn, quarantine

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Flags each row with the rules it breaks, splits good and quarantined rows, deduplicates, and fails the step if any <b>fail</b>-severity check is broken.<br><br>
<b>Why it matters</b><br>Checks are only useful if they change what happens. Row-level rules route bad rows to quarantine <b>with the reason</b>, so they can be fixed and replayed. Severe problems stop the pipeline before bad data is written.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The step reports the failing <code>fail</code>-severity checks (uniqueness and not-null on <code>customer_id</code>) and stops. With <code>enforce=False</code>, it shows 4 good rows and 5 quarantined rows, each with a list of reasons.
</div>

In [0]:
row_rules = {
    "customer_id is null": F.col("customer_id").isNull(),
    "amount out of range": ~F.col("amount").between(0, 100_000),
    "invalid email": F.col("email").isNotNull() & ~F.col("email").rlike(EMAIL),
    "unknown status": ~F.col("status").isin(STATUSES),
}

def apply_quality(df, customers_df, results, enforce=True):
    blocking = [f"{r.check}({r.column})" for r in results if r.severity == "fail" and not r.passed]
    if blocking and enforce:
        raise ValueError(f"blocking data quality checks failed: {blocking}")

    deduped = df.dropDuplicates(["order_id"])
    known = customers_df.withColumn("_known", F.lit(True))
    with_fk = deduped.join(known, "customer_id", "left")
    reasons = F.array_compact(F.array(*[F.when(cond, F.lit(name)) for name, cond in row_rules.items()]
                                      + [F.when(F.col("customer_id").isNotNull() & F.col("_known").isNull(), F.lit("unknown customer"))]))
    flagged = with_fk.withColumn("dq_reasons", reasons).drop("_known")
    good = flagged.filter(F.size("dq_reasons") == 0).drop("dq_reasons")
    quarantine = flagged.filter(F.size("dq_reasons") > 0)
    return good, quarantine

try:
    apply_quality(orders, customers, results)
except ValueError as e:
    print("STOPPED:", e)

good, quarantine = apply_quality(orders, customers, results, enforce=False)
print("good rows:", good.count())
good.select("order_id", "customer_id", "amount", "status").orderBy("order_id").show()
quarantine.select("order_id", "customer_id", "amount", "status", "dq_reasons").orderBy("order_id").show(truncate=False)

## 6. Constraints in the table itself (Delta preview)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds <code>NOT NULL</code> and <code>CHECK</code> constraints to a Delta table, then tries to insert a row that breaks them.<br><br>
<b>Why it matters</b><br>Constraints enforce rules <b>at write time</b> for every writer, not just your pipeline. The whole write is rejected if any row breaks a rule. Covered in <code>03_delta_lake/12_constraints_generated_columns</code>. Lakeflow Declarative Pipelines add <b>expectations</b> with warn, drop and fail actions, covered in <code>07_workflows</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The valid insert succeeds. The insert with a negative amount fails with a <code>DELTA_VIOLATE_CONSTRAINT_WITH_VALUES</code> error naming the <code>positive_amount</code> constraint.
</div>

In [0]:
CATALOG, SCHEMA = "workspace", "spark_basics"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
DB = f"{CATALOG}.{SCHEMA}"

In [0]:
table = f"{DB}.lesson29_orders"
spark.sql(f"CREATE OR REPLACE TABLE {table} (order_id STRING NOT NULL, amount DOUBLE) USING DELTA")
spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT positive_amount CHECK (amount >= 0)")

spark.sql(f"INSERT INTO {table} VALUES ('O1', 120.0)")
print("valid insert OK")

try:
    spark.sql(f"INSERT INTO {table} VALUES ('O2', -5.0)")
except Exception as e:
    print("rejected:", type(e).__name__, "-", str(e).splitlines()[0][:150])

spark.sql(f"DROP TABLE IF EXISTS {table}")

## Under the hood

```
check_not_null + check_in_range + ...  as separate functions
   -> each calls an action (agg / count) -> one Spark job per check

one df.agg(sum(when(rule1,1)), sum(when(rule2,1)), ...)
   -> ONE scan computes every rule at once
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> every check that returns a number runs a job. Many small checks on a large table mean many full scans.

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> row-level flagging (`withColumn` + `when`) is narrow and cheap.

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> uniqueness (`distinct`) and foreign-key checks (anti join) shuffle.

**Production tip:** for big tables, compute all row-level rules as columns **once**, aggregate them in a single `agg`, and write the flagged result. The code below computes every rule's failure count in one pass, which is what the reusable functions above trade away for readability.

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
One row with the failure count of each row rule, from a single aggregation.
</div>

In [0]:
orders.agg(*[F.sum(F.when(cond, 1).otherwise(0)).alias(name.replace(" ", "_")) for name, cond in row_rules.items()]).show()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: checks pass but the data is still wrong</b><br>
Null comparisons return null and are counted as "not failing". Write rules that handle nulls explicitly (<code>isNull() | ...</code>).<br><br>
<b>⛔ Problem: data quality checks make the pipeline slow</b><br>Each check is a separate job. Combine rules into one aggregation and flag rows once.<br><br>
<b>⛔ Problem: quarantined rows are never looked at</b><br>Store them in a table with the reasons and a timestamp, monitor the count, and alert on it.<br><br>
<b>⛔ Problem: every failure stops the pipeline</b><br>Not every rule is critical. Assign severities: fail for keys and schema, quarantine for bad rows, warn for drift.<br><br>
<b>⛔ Problem: checks run after the write</b><br>Run checks <b>before</b> writing to the target table, or write to a staging table and promote only after checks pass.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What data quality checks would you put in a pipeline?</b><br>
Completeness (required columns not null), uniqueness of keys, validity (allowed values and formats), ranges, referential integrity against dimension tables, freshness, volume compared with history, and schema against a contract.<br><br>

<b>🎤 2. What do you do with rows that fail a check?</b><br>
It depends on severity. Critical failures such as duplicate keys or a broken schema stop the pipeline. Row-level problems go to a quarantine table with the reason, so good rows continue and bad ones can be fixed and replayed. Minor drift raises a warning.<br><br>

<b>🎤 3. How do you make data quality checks efficient on big data?</b><br>
Express rules as column expressions and evaluate them all in one aggregation, instead of one job per check. Flag rows once with a reasons column, then split good and bad rows from that.<br><br>

<b>🎤 4. What is the difference between a Delta constraint and a pipeline check?</b><br>
A Delta constraint (<code>CHECK</code>, <code>NOT NULL</code>) is enforced by the table on every write, by any writer, and rejects the whole transaction if a row violates it. A pipeline check is code you run, which can warn, quarantine or fail with more flexible logic.<br><br>

<b>🎤 5. What are expectations in Lakeflow Declarative Pipelines?</b><br>
Declarative data quality rules on pipeline datasets, with three actions: keep and record violations (warn), drop violating rows, or fail the update. Results are recorded in the pipeline's event log.<br><br>

<b>🎤 6. How do you check freshness?</b><br>
Compare the maximum event or ingestion timestamp with the current time and alert if it is older than the agreed SLA.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A Delta table must reject any write containing a negative <code>amount</code>. What should the data engineer use?</b><br>
A. A view that filters out negative amounts<br>
B. <code>ALTER TABLE t ADD CONSTRAINT positive CHECK (amount &gt;= 0)</code><br>
C. A comment on the column<br>
D. <code>OPTIMIZE</code> with a filter<br>
<details><summary><b>Show answer</b></summary><b>B.</b> A <code>CHECK</code> constraint is enforced on every write, and the transaction fails if any row violates it.</details><br><br>

<b>Q2. In Lakeflow Declarative Pipelines, which expectation action removes invalid rows but lets the update continue?</b><br>
A. <code>EXPECT ... ON VIOLATION FAIL UPDATE</code><br>
B. <code>EXPECT ... ON VIOLATION DROP ROW</code><br>
C. <code>EXPECT ...</code> with no action<br>
D. <code>EXPECT ... ON VIOLATION QUARANTINE</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> No action keeps the rows and only records violations. <code>FAIL UPDATE</code> stops the update. There is no built-in <code>QUARANTINE</code> action, so quarantine is built with a separate dataset using the inverse rule.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Add a check <code>check_max_age(df, ts_col, max_hours, now)</code> that returns a <code>CheckResult</code> with 1 failed row if the newest timestamp is older than <code>max_hours</code></li><li>Add a rule that flags orders where <code>status = 'REFUND'</code> but <code>amount</code> is positive, and include it in <code>row_rules</code></li><li>Re-run <code>apply_quality(..., enforce=False)</code> and show the new quarantine with reasons</li><li>Save the check results (<code>results_df</code>) to a Delta table <code>lesson29_dq_results</code> in append mode, so results can be tracked over time</li><li>In one sentence: which checks in this lesson would you mark as <code>fail</code>, and why?</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Freshness as a reusable check
def check_max_age(df, ts_col, max_hours, now, severity="warn"):
    latest = df.agg(F.max(ts_col)).first()[0]
    stale = latest is None or (now - latest).total_seconds() / 3600 > max_hours
    return CheckResult(f"max_age<={max_hours}h", ts_col, int(stale), df.count(), severity)

print(check_max_age(orders, "order_ts", 24, now))
print(check_max_age(orders, "order_ts", 1, now))

# 2. A cross-column rule: refunds should be negative.
#    No sample row breaks it (O4 is a refund of -15), so the quarantine below doesn't change.
row_rules["refund with positive amount"] = (F.col("status") == "REFUND") & (F.col("amount") > 0)

# 3. Re-run with the extra rule
good2, quarantine2 = apply_quality(orders, customers, results, enforce=False)
quarantine2.select("order_id", "status", "amount", "dq_reasons").orderBy("order_id").show(truncate=False)

# 4. Keep results over time
results_df.write.mode("append").saveAsTable(f"{DB}.lesson29_dq_results")
spark.table(f"{DB}.lesson29_dq_results").show(truncate=False)
spark.sql(f"DROP TABLE IF EXISTS {DB}.lesson29_dq_results")

# 5. Uniqueness of the key and the schema contract: they break joins and every downstream table,
#    so continuing would spread the problem. Row-level issues can be quarantined instead.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Dimensions: completeness, uniqueness, validity, range, integrity, freshness, volume, schema</li><li>Write checks as reusable functions that return a standard result, and store the results</li><li>Actions: <b>fail</b> (keys, schema), <b>quarantine</b> (bad rows with reasons), <b>warn</b> (drift)</li><li>Combine rules into one aggregation for performance. Flag rows once with a reasons array</li><li>Handle nulls explicitly in rules</li><li>Delta <code>CHECK</code> / <code>NOT NULL</code> constraints enforce rules at write time</li><li>Lakeflow expectations: warn (default), <code>DROP ROW</code>, <code>FAIL UPDATE</code></li></ul>
</div>

| Task | Code |
|---|---|
| Null count | `F.sum(F.col(c).isNull().cast("int"))` |
| Duplicates | `df.count() - df.select(k).distinct().count()` |
| Range | `~F.col(c).between(lo, hi)` |
| Pattern | `~F.col(c).rlike(pattern)` |
| Orphans | `df.join(dim, key, "left_anti").count()` |
| Reasons array | `F.array_compact(F.array(*[F.when(cond, F.lit(name)) ...]))` |
| Delta constraint | `ALTER TABLE t ADD CONSTRAINT c CHECK (amount >= 0)` |

## Git commit

```
git add 01_spark_basics/29_data_quality_checks.ipynb
git commit -m "add 29 data quality checks notebook"
```